In [1]:
#imports
import pandas as pd

In [2]:
#CSV - loan.csv
df_csv = pd.read_csv("./dataset/loan.csv", low_memory=False)

#XLSX - LCDataDictionary
df_xlsx = pd.read_excel("./dataset/LCDataDictionary.xlsx")

ImportError: `Import openpyxl` failed.  Use pip, conda, or your preferred package management tool to install the openpyxl package.

In [ ]:
#Printe a boere of the data
print("Before:", df_csv.shape)

In [ ]:
#show the lodaed data:
#df_csv.shape
df_csv.head()
#df_csv.columns.tolist()

In [ ]:
#Data Cleaning 
drop_cols = ['member_id', 'id','emp_title','url','desc','title','zip_code','policy_code',
 'funded_amnt_inv','out_prncp_inv','total_pymnt_inv',
 'last_pymnt_d','last_pymnt_amnt','next_pymnt_d','last_credit_pull_d',
 'annual_inc_joint','dti_joint','verification_status_joint','revol_bal_joint',
 'sec_app_earliest_cr_line','sec_app_inq_last_6mths','sec_app_mort_acc',
 'sec_app_open_acc','sec_app_revol_util','sec_app_open_act_il',
 'sec_app_num_rev_accts','sec_app_chargeoff_within_12_mths',
 'sec_app_collections_12_mths_ex_med','sec_app_mths_since_last_major_derog',
 'hardship_flag','hardship_type','hardship_reason','hardship_status','deferral_term',
 'hardship_amount','hardship_start_date','hardship_end_date','payment_plan_start_date',
 'hardship_length','hardship_dpd','hardship_loan_status',
 'orig_projected_additional_accrued_interest','hardship_payoff_balance_amount',
 'hardship_last_payment_amount','debt_settlement_flag','debt_settlement_flag_date',
 'settlement_status','settlement_date','settlement_amount','settlement_percentage',
 'settlement_term', 'installment', 
 'out_prncp', 'total_rec_prncp', 'total_rec_int', 
 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee']

clean_df = df_csv.drop(columns=drop_cols).copy()

# Only keep loans that reached a final state. This keeps our model training on loans that have completed or defaulted. 
final_status = ['Fully Paid', 'Charged Off', 'Default',
                'Late (31-120 days)', 'Late (16-30 days)', 'In Grace Period']
clean_df = clean_df[clean_df['loan_status'].isin(final_status)].copy()

#drop loan status. a paid loan will be profitable usually
clean_df = clean_df.drop(columns='loan_status').copy()

clean_df.shape

In [ ]:
#Now our feature we want. Is this loan profitable? profitable = total_payment > funded_amnt
clean_df['profitable'] = (clean_df['total_pymnt'] > clean_df['funded_amnt']).astype(int)
# Check the balance of your target
clean_df['profitable'].value_counts()

In [ ]:
#Dropping leaky column. 
leaky_columns = ['total_pymnt', 'funded_amnt']
clean_df = clean_df.drop(columns=leaky_columns).copy()

In [ ]:
clean_df.head()

In [ ]:
#Show non numerical vlaues and what they look like. 
non_numeric = [c for c in clean_df.columns if not pd.api.types.is_numeric_dtype(clean_df[c])]
for c in non_numeric:
    print(c, clean_df[c].dtype)

for c in non_numeric:
    print("=== ", c, " ===")
    print(clean_df[c].value_counts(dropna=False).head(10))
    print()

In [ ]:
clean_df.shape

In [ ]:
#Remove the term months into just an integer of months
clean_df['term_months'] = clean_df['term'].str.extract(r'(\d+)').astype(int)
clean_df = clean_df.drop(columns=['term'])

In [ ]:
#Grade and subgrade get mapped to integer values, with A = 0 (best credit) G = 6 (worst credit)
grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
clean_df['grade'] = clean_df['grade'].map(grade_map)

sub_grade_map = {
    'A1':0,'A2':1,'A3':2,'A4':3,'A5':4,
    'B1':5,'B2':6,'B3':7,'B4':8,'B5':9,
    'C1':10,'C2':11,'C3':12,'C4':13,'C5':14,
    'D1':15,'D2':16,'D3':17,'D4':18,'D5':19,
    'E1':20,'E2':21,'E3':22,'E4':23,'E5':24,
    'F1':25,'F2':26,'F3':27,'F4':28,'F5':29,
    'G1':30,'G2':31,'G3':32,'G4':33,'G5':34
}
clean_df['sub_grade'] = clean_df['sub_grade'].map(sub_grade_map)

In [ ]:
#cleaning up employment length
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}
clean_df['emp_length'] = clean_df['emp_length'].map(emp_map)

In [ ]:
#cleaning up home_ownership,  purpose, addr_state, verification_status 
#one-hot was used to give a binary values to indacte the value of the feature. 
#numerical values are not used so that the model deos attempt to asscoaite. OWN > MORTGAGE > RENT. 3 > 2 > 1

# Binary columns -> simple 0/1
clean_df['pymnt_plan'] = (clean_df['pymnt_plan'] == 'y').astype(int)
clean_df['initial_list_status'] = (clean_df['initial_list_status'] == 'f').astype(int)
clean_df['application_type'] = (clean_df['application_type'] == 'Joint App').astype(int)
clean_df['disbursement_method'] = (clean_df['disbursement_method'] == 'DirectPay').astype(int)

# Multi-category nominal columns -> one-hot
clean_df = pd.get_dummies(clean_df,
                          columns=['home_ownership','purpose','addr_state','verification_status'],
                          drop_first=True, dtype=int)

In [ ]:
# --- Date handling: issue_d and earliest_cr_line ---

# Parse the Mon-YYYY strings into proper datetime objects
clean_df['issue_d'] = pd.to_datetime(clean_df['issue_d'], format='%b-%Y')
clean_df['earliest_cr_line'] = pd.to_datetime(clean_df['earliest_cr_line'], format='%b-%Y')

# Extract the issue year (loan vintage) - useful because risk varies by economic era
clean_df['issue_year'] = clean_df['issue_d'].dt.year

# Credit history length: how many years between oldest credit line and this loan
clean_df['credit_history_years'] = clean_df['issue_d'].dt.year - clean_df['earliest_cr_line'].dt.year

# Fill the few nulls (from missing earliest_cr_line) with the median
clean_df['credit_history_years'] = clean_df['credit_history_years'].fillna(
    clean_df['credit_history_years'].median()
)

# Drop the raw datetime columns now that we've extracted what we need
clean_df = clean_df.drop(columns=['issue_d', 'earliest_cr_line'])

# Verify
clean_df['credit_history_years'].describe()

In [ ]:
#Show remainging non numerical vlaues and what they look like. 
non_numeric = [c for c in clean_df.columns if not pd.api.types.is_numeric_dtype(clean_df[c])]
for c in non_numeric:
    print(c, clean_df[c].dtype)

for c in non_numeric:
    print("=== ", c, " ===")
    print(clean_df[c].value_counts(dropna=False).head(10))
    print("")
print("END")

In [ ]:
clean_df.head()

In [ ]:
clean_df.shape

In [ ]:
#Check is all rows have a grade.
missing = clean_df[clean_df['grade'].notna() & clean_df['sub_grade'].isna()]
print("Rows with grade but no sub_grade:", len(missing))

missing_rev = clean_df[clean_df['sub_grade'].notna() & clean_df['grade'].isna()]
print("Rows with sub_grade but no grade:", len(missing_rev))

In [ ]:
# Save the cleaned DataFrame to a CSV file
# clean_df.to_csv("~/Optimizing-P2P-Loan-Thresholds-for-Borrowers-Using-LendingClub-Data-/dataset/clean_loans.csv", index=False)

In [ ]:
# Correlate every column with the target
# corr = clean_df.corr()['profitable'].drop('profitable').sort_values(key=abs, ascending=False)
# corr

In [ ]:
# Undersampling hte majority case
# Want to create a balnced data set for cleaning

df_profit = clean_df[clean_df['profitable'] == 1]
df_loss = clean_df[clean_df['profitable'] == 0]

# ratio of profit vs. loss represented in the sampled dataset.
samplingRatio = 

# create a new DataFrame with the same number of profitable loans as unprofitable loans
df_profit_undersampled = df_profit.sample(n=len(df_loss), random_state=42)

# combine the undersampled profits with the origianl losses
df_undersampled_clean = pd.concat([df_profit_undersampled, df_loss])

# Shuffle the rows of the new DataFrame to mix the profitable and unprofitable loans
df_undersampled_clean = df_undersampled_clean.sample(frac=1, random_state=42).reset_index(drop=True)

df_undersampled_clean.shape

In [3]:
# Save the cleaned DataFrame to a CSV file
df_undersampled_clean.to_csv("~/Optimizing-P2P-Loan-Thresholds-for-Borrowers-Using-LendingClub-Data-/dataset/50-50clean_undersampled.csv", index=False)
print("undersampled csv built")

NameError: name 'df_undersampled_clean' is not defined